# 08 Analyse, Visualisierung und Ergebnisgeschichte

## Zweck
Dieses Notebook übersetzt die geprüften Gold-Daten aus Phase 7 in eine stringente, deutschsprachige Ergebnisgeschichte. Der Fokus liegt auf nachvollziehbaren deskriptiven Aussagen und einer explorativen Kontextanalyse.

**Analysevertrag:** Die Auswertung beschreibt den betrachteten Datensatz. Sie beweist keine Kausalität, keine allgemeingültige Rangfolge europäischer Städte und keinen langfristigen Trend aus einer einzelnen Live-Momentaufnahme.

## Forschungsfrage und Hypothesen

**Leitfrage:** Welche Luftqualitätsmuster zeigen ausgewählte europäische Städte im historischen Vergleich, und wie lassen sich diese Unterschiede durch urbane Kontextdaten vorsichtig einordnen?

**H1:** Die mittleren historischen PM2.5-Werte unterscheiden sich zwischen den betrachteten Städten.

**H2:** Die Rangfolge der Städte ist schadstoffabhängig; PM2.5, PM10 und NO2 müssen getrennt interpretiert werden.

**H3:** Bevölkerungsdichte kann als Kontextvariable explorativ mit Luftqualitätskennzahlen zusammenhängen. Sie ist kein kausaler Erklärungsfaktor.

**H4:** Die Open-Meteo-Live-Werte ergänzen die Architektur als Momentaufnahme. Sie dürfen nicht mit historischen EEA-Tagesaggregaten gleichgesetzt werden.

## Eingaben und Ausgaben

Eingaben sind ausschließlich die fünf Gold-Parquets aus Notebook `07`. Das Notebook erzeugt sechs Abbildungen unter `presentation/figures/` und verwendet keine Bronze- oder Silver-Daten für die Analyse.

## Technologien
Python, pandas, NumPy, Matplotlib, Parquet und Markdown.

Die Struktur setzt auf Datenprofil vor Interpretation, nachvollziehbare Gruppierungen, sichtbare Beobachtungszahlen, gespeicherte Abbildungen und vorsichtige Aussagen.

## Konfiguration

### Projektpfade auflösen
Die Pfade werden vom Repository-Root abgeleitet. Dadurch funktioniert das Notebook sowohl aus dem Projektordner als auch aus dem Unterordner `notebooks/`.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
load_dotenv(PROJECT_ROOT / ".env")

DATA_DIR = PROJECT_ROOT / Path(os.getenv("DATA_DIR", "data"))
GOLD_DIR = DATA_DIR / "gold"
FIGURE_DIR = PROJECT_ROOT / "presentation" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
DISPLAY_UNIT = "µg/m³"
print({"project_root": str(PROJECT_ROOT), "figure_dir": str(FIGURE_DIR)})

### Gold-Verträge definieren

Phase 8 arbeitet bewusst nur auf analysefertigen Gold-Dateien. Fehlende Dateien stoppen die Auswertung mit einer klaren Fehlermeldung.

In [ ]:
GOLD_PATHS = {
    "daily": GOLD_DIR / "city_air_quality_daily_summary.parquet",
    "ranking": GOLD_DIR / "pollutant_ranking_by_city.parquet",
    "context": GOLD_DIR / "city_context_air_quality.parquet",
    "live": GOLD_DIR / "live_air_quality_latest.parquet",
    "quality": GOLD_DIR / "data_quality_summary.parquet",
}
for name, path in GOLD_PATHS.items():
    assert path.exists(), f"Gold-Eingabe fehlt: {name} -> {path}"
assert not (PROJECT_ROOT / "notebooks" / "data").exists(), "Falscher Ausgabeordner notebooks/data gefunden"
print("Alle Gold-Eingaben sind vorhanden.")

## Datenprofil

### Gold-Dateien laden
Die fünf Gold-Tabellen werden eingelesen und ihre Größen sichtbar ausgegeben. Das ist die Grundlage für jede spätere Interpretation.

In [ ]:
daily_df = pd.read_parquet(GOLD_PATHS["daily"])
ranking_df = pd.read_parquet(GOLD_PATHS["ranking"])
context_df = pd.read_parquet(GOLD_PATHS["context"])
live_df = pd.read_parquet(GOLD_PATHS["live"])
quality_df = pd.read_parquet(GOLD_PATHS["quality"])

gold_frames = {
    "Historische Tageswerte": daily_df,
    "Städterankings": ranking_df,
    "Stadtkontext": context_df,
    "Live-Momentaufnahme": live_df,
    "Qualitätsbericht": quality_df,
}
pd.DataFrame([
    {"Datensatz": name, "Zeilen": len(frame), "Spalten": len(frame.columns)}
    for name, frame in gold_frames.items()
])

### Historischen Analyseumfang bestimmen

Zeitraum, Städtezahl, Schadstoffe, Quellen und Beobachtungszahlen werden vor dem Plotting offengelegt.

In [ ]:
daily_df["date"] = pd.to_datetime(daily_df["date"])
historical_start = daily_df["date"].min().date()
historical_end = daily_df["date"].max().date()
historical_statuses = sorted(daily_df["data_status"].unique())
pollutants = sorted(daily_df["pollutant"].unique())

historical_scope = {
    "Zeitraum": f"{historical_start} bis {historical_end}",
    "Städte": int(daily_df["city_id"].nunique()),
    "Schadstoffe": pollutants,
    "Tageszeilen": len(daily_df),
    "Quellen": sorted(daily_df["source"].unique()),
    "Datenstatus": historical_statuses,
}
historical_scope

### Datenqualität und Aussagegrenzen prüfen

Der lokale Kontrollsample bleibt sichtbar. Nur ein realer EEA-Extract erlaubt finale empirische Aussagen. Live-Daten werden separat als Momentaufnahme markiert.

In [ ]:
eea_sample_fallback_used = set(historical_statuses) != {"real_eea_file"}
live_input_modes = sorted(live_df["live_input_mode"].unique())
live_event_start = pd.to_datetime(live_df["event_time_ts"], utc=True).min()
live_event_end = pd.to_datetime(live_df["event_time_ts"], utc=True).max()

analysis_guardrails = {
    "Finale historische Aussagen zulässig": not eea_sample_fallback_used,
    "EEA-Sample-Fallback aktiv": eea_sample_fallback_used,
    "Live-Eingabemodus": live_input_modes,
    "Live-Zeitraum": f"{live_event_start} bis {live_event_end}",
    "Hinweis": "Lokale Ergebnisse sind Demo-Ergebnisse, solange kontrollierte Fallback-Daten aktiv sind.",
}
analysis_guardrails

### Beobachtungszahlen und fehlende Werte untersuchen

Rankings sind nur sinnvoll, wenn ihre Datengrundlage sichtbar ist. Deshalb werden Beobachtungszahlen pro Stadt und Schadstoff sowie fehlende Werte ausgegeben.

In [ ]:
observation_profile = (
    daily_df.groupby(["city_name", "pollutant"], as_index=False)
    .agg(
        Tageswerte=("date", "count"),
        Messwerte=("measurement_count", "sum"),
        Mittelwert=("avg_value", "mean"),
        Median=("avg_value", "median"),
        Standardabweichung=("avg_value", "std"),
        Minimum=("min_value", "min"),
        Maximum=("max_value", "max"),
    )
)
missing_profile = pd.DataFrame({
    "Datensatz": list(gold_frames),
    "Fehlende Werte": [int(frame.isna().sum().sum()) for frame in gold_frames.values()],
})
display(observation_profile.head(12))
missing_profile

## Visualisierung 1: PM2.5-Städteranking

### Ranking vorbereiten
Für H1 werden mittlere PM2.5-Werte absteigend sortiert. Die Abbildung zeigt zusätzlich die Zahl der historischen Tageswerte pro Stadt.

In [ ]:
pm25_ranking = (
    observation_profile.query("pollutant == 'pm2_5'")
    .sort_values("Mittelwert", ascending=True)
    .copy()
)
assert len(pm25_ranking) == daily_df["city_id"].nunique()
pm25_ranking[["city_name", "Mittelwert", "Median", "Tageswerte", "Messwerte"]]

### PM2.5-Ranking zeichnen

Die Darstellung ist deskriptiv. Bei aktivem Sample-Fallback dient sie nur als reproduzierbare Demo der Analysemechanik.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(pm25_ranking["city_name"], pm25_ranking["Mittelwert"], color="#2f6690")
for bar, n in zip(bars, pm25_ranking["Tageswerte"]):
    ax.text(bar.get_width() + 0.08, bar.get_y() + bar.get_height()/2, f"n={n}", va="center", fontsize=9)
ax.set_title(f"Mittlere PM2.5-Belastung nach Stadt\nEEA-Tageswerte, {historical_start} bis {historical_end}")
ax.set_xlabel(f"PM2.5-Mittelwert ({DISPLAY_UNIT})")
ax.set_ylabel("Stadt")
fig.text(0.01, 0.01, "Quelle: EEA | Deskriptiver Vergleich | n = Anzahl Tageswerte | Sample-Fallback möglich", fontsize=8)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(FIGURE_DIR / "pm25_city_ranking.png", dpi=160)
plt.show()

## Visualisierung 2: Schadstoffvergleich

### Vergleichstabelle erzeugen
Für H2 werden PM2.5, PM10 und NO2 getrennt ausgewertet. Gleiche Einheiten erlauben eine gemeinsame Achse, aber keine Aussage darüber, welcher Schadstoff gesundheitlich „schlimmer“ ist.

In [ ]:
pollutant_labels = {"pm2_5": "PM2.5", "pm10": "PM10", "no2": "NO2"}
comparison_df = ranking_df.copy()
comparison_df["Schadstoff"] = comparison_df["pollutant"].map(pollutant_labels)
comparison_pivot = comparison_df.pivot(index="city_name", columns="Schadstoff", values="mean_pollutant_value")
comparison_pivot = comparison_pivot[["PM2.5", "PM10", "NO2"]].sort_values("PM2.5")
comparison_pivot

### Gruppierten Balkenplot zeichnen

Der Plot zeigt schadstoffspezifische Muster. Die Interpretation bleibt auf Unterschiede innerhalb des betrachteten Datensatzes begrenzt.

In [ ]:
ax = comparison_pivot.plot(kind="barh", figsize=(11, 7), color=["#2f6690", "#f4a261", "#6a994e"])
ax.set_title(f"Schadstoffvergleich nach Stadt\nEEA-Tageswerte, {historical_start} bis {historical_end}")
ax.set_xlabel(f"Mittlere Konzentration ({DISPLAY_UNIT})")
ax.set_ylabel("Stadt")
ax.legend(title="Schadstoff")
ax.figure.text(0.01, 0.01, "Quelle: EEA | Schadstoffe fachlich getrennt interpretieren", fontsize=8)
ax.figure.tight_layout(rect=(0, 0.04, 1, 1))
ax.figure.savefig(FIGURE_DIR / "pollutant_comparison.png", dpi=160)
plt.show()

## Visualisierung 3: Zeitreihe ausgewählter Städte

### Vergleichsstädte auswählen
Für PM2.5 werden die Stadt mit dem höchsten, die Stadt mit dem niedrigsten und eine mittlere Position ausgewählt. So bleibt die Auswahl nachvollziehbar.

In [ ]:
pm25_desc = pm25_ranking.sort_values("Mittelwert", ascending=False).reset_index(drop=True)
selected_cities = [
    pm25_desc.iloc[0]["city_name"],
    pm25_desc.iloc[len(pm25_desc)//2]["city_name"],
    pm25_desc.iloc[-1]["city_name"],
]
selected_cities

### PM2.5-Zeitreihe zeichnen

Die Tageswerte werden ohne Interpolation und ohne Glättung dargestellt. Bei einem kurzen Zeitraum wird ausdrücklich kein langfristiger Trend behauptet.

In [ ]:
timeseries_df = daily_df.query("pollutant == 'pm2_5' and city_name in @selected_cities").copy()
fig, ax = plt.subplots(figsize=(11, 6))
for city, group in timeseries_df.groupby("city_name"):
    ax.plot(group["date"], group["avg_value"], marker="o", markersize=3, linewidth=1.5, label=city)
ax.set_title(f"PM2.5-Tageswerte ausgewählter Städte\n{historical_start} bis {historical_end}, ohne Interpolation")
ax.set_xlabel("Datum")
ax.set_ylabel(f"PM2.5-Tagesmittel ({DISPLAY_UNIT})")
ax.legend(title="Stadt")
fig.text(0.01, 0.01, "Quelle: EEA | Kurzer Betrachtungszeitraum: keine langfristige Trendaussage", fontsize=8)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(FIGURE_DIR / "selected_city_timeseries.png", dpi=160)
plt.show()

## Visualisierung 4: PM2.5-Verteilungen

### Beobachtungszahl für Boxplots prüfen
Boxplots sind hier sinnvoll, weil jede Stadt ausreichend Tageswerte besitzt. Ausreißer bleiben sichtbar und werden nicht entfernt.

In [ ]:
pm25_daily = daily_df.query("pollutant == 'pm2_5'").copy()
pm25_counts = pm25_daily.groupby("city_name")["date"].count().sort_values()
assert pm25_counts.min() >= 10, "Zu wenige Tageswerte für Boxplots; alternativ Punktplot verwenden."
pm25_counts

### Boxplots zeichnen

Median, Streuung und sichtbare Ausreißer ergänzen die Mittelwert-Rankings. Streuung ist nicht mit statistischer Unsicherheit gleichzusetzen.

In [ ]:
city_order = pm25_counts.index.tolist()
box_data = [pm25_daily.loc[pm25_daily["city_name"] == city, "avg_value"] for city in city_order]
fig, ax = plt.subplots(figsize=(11, 6))
ax.boxplot(box_data, tick_labels=city_order, showfliers=True)
ax.set_title(f"Verteilung der PM2.5-Tagesmittel\nEEA, {historical_start} bis {historical_end}")
ax.set_xlabel("Stadt")
ax.set_ylabel(f"PM2.5-Tagesmittel ({DISPLAY_UNIT})")
ax.tick_params(axis="x", rotation=35)
fig.text(0.01, 0.01, "Quelle: EEA | Ausreißer bleiben sichtbar | Streuung ist keine Signifikanzprüfung", fontsize=8)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(FIGURE_DIR / "pollutant_distribution.png", dpi=160)
plt.show()

## Visualisierung 5: Bevölkerungsdichte als Kontextvariable

### Explorative Spearman-Korrelation berechnen
Für H3 wird PM2.5 gegen Bevölkerungsdichte dargestellt. Bei nur acht Städten ist Spearman lediglich ein explorativer Hinweis, keine belastbare Ursache-Wirkungs-Aussage.

In [ ]:
density_df = context_df.query("pollutant == 'pm2_5'").dropna(subset=["population_density", "mean_pollutant_value"]).copy()
density_df["Dichte_Rang"] = density_df["population_density"].rank()
density_df["PM25_Rang"] = density_df["mean_pollutant_value"].rank()
spearman_rho = density_df["Dichte_Rang"].corr(density_df["PM25_Rang"], method="pearson")
density_context = {
    "Methode": "Spearman-Rangkorrelation",
    "n_Städte": len(density_df),
    "rho": round(float(spearman_rho), 3),
    "Interpretation": "explorativ, nicht kausal",
}
density_context

### Kontextplot zeichnen

Stadtlabels machen die kleine Stichprobe transparent. Relevante Confounder sind unter anderem Verkehr, Industrie, Wetter, Topografie, Messstandorte und Maßnahmenpolitik.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(density_df["population_density"], density_df["mean_pollutant_value"], s=65, color="#9b2226")
for _, row in density_df.iterrows():
    ax.annotate(row["city_name"], (row["population_density"], row["mean_pollutant_value"]), xytext=(5, 4), textcoords="offset points", fontsize=9)
ax.set_title(f"Bevölkerungsdichte und mittlere PM2.5-Belastung\nExplorativ: Spearman ρ={spearman_rho:.2f}, n={len(density_df)} Städte")
ax.set_xlabel("Bevölkerungsdichte (Einwohner je km²)")
ax.set_ylabel(f"Mittlere PM2.5-Konzentration ({DISPLAY_UNIT})")
fig.text(0.01, 0.01, "Quellen: EEA, Wikipedia | Explorativ, nicht kausal | Confounder nicht kontrolliert", fontsize=8)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(FIGURE_DIR / "density_vs_air_quality.png", dpi=160)
plt.show()

## Visualisierung 6: getrennte Open-Meteo-Live-Momentaufnahme

### Live-Daten für den Plot vorbereiten
Die Live-Werte werden separat gezeigt. Sie sind technisch relevant für den Kafka-/Spark-Pfad, aber historisch nicht repräsentativ.

In [ ]:
live_plot_df = live_df[["city_name", "pm2_5", "pm10", "no2", "event_time_ts", "data_status", "live_input_mode"]].copy()
live_plot_df = live_plot_df.sort_values("pm2_5")
live_time_label = f"{live_event_start} bis {live_event_end}"
live_plot_df

### Live-Snapshot zeichnen

Die Abbildung ist ausdrücklich als Momentaufnahme beschriftet. Ein kurzfristiges Ranking darf nicht als langfristige Luftqualitätsrangfolge interpretiert werden.

In [ ]:
live_chart = live_plot_df.set_index("city_name")[["pm2_5", "pm10", "no2"]].rename(columns=pollutant_labels)
ax = live_chart.plot(kind="barh", figsize=(11, 7), color=["#2f6690", "#f4a261", "#6a994e"])
ax.set_title(f"Open-Meteo-Live-Momentaufnahme\nEvent-Zeit: {live_time_label}")
ax.set_xlabel(f"Konzentration ({DISPLAY_UNIT})")
ax.set_ylabel("Stadt")
ax.legend(title="Schadstoff")
ax.figure.text(0.01, 0.01, "Quelle: Open-Meteo | Momentaufnahme, nicht historisch repräsentativ | Live-Fallback möglich", fontsize=8)
ax.figure.tight_layout(rect=(0, 0.04, 1, 1))
ax.figure.savefig(FIGURE_DIR / "live_air_quality_snapshot.png", dpi=160)
plt.show()

## Validierung und Qualitätschecks

### Figuren und methodische Trennung prüfen
Die abschließenden Assertions stellen sicher, dass alle sechs Figuren existieren und historische sowie aktuelle Kontexte getrennt bleiben.

In [ ]:
FIGURES = [
    "pm25_city_ranking.png",
    "pollutant_comparison.png",
    "selected_city_timeseries.png",
    "pollutant_distribution.png",
    "density_vs_air_quality.png",
    "live_air_quality_snapshot.png",
]
for name in FIGURES:
    path = FIGURE_DIR / name
    assert path.exists() and path.stat().st_size > 0, f"Figur fehlt oder ist leer: {path}"
assert set(daily_df["dataset_context"]) == {"eea_historical"}
assert set(live_df["dataset_context"]) == {"open_meteo_live"}
assert not (PROJECT_ROOT / "notebooks" / "data").exists()
print({"figures": FIGURES, "historical_live_separation": "PASS"})

## Ergebnisse

Die lokale Ausführung erzeugt eine vollständige, reproduzierbare Analysegeschichte. Solange `controlled_sample_fallback` aktiv ist, sind die historischen Diagramme methodische Demonstrationen. Nach Einspielen eines realen EEA-Extracts können dieselben Schritte unverändert für empirische Aussagen genutzt werden.

Die Live-Abbildung bleibt in jedem Fall eine getrennte Open-Meteo-Momentaufnahme. Die Kontextanalyse zur Bevölkerungsdichte ist explorativ und nicht kausal.

## Limitationen

- Historische Aussagen benötigen einen realen EEA-Extract.
- Open-Meteo-Live-Werte sind Momentaufnahmen.
- Wikipedia-Metadaten stammen aus heuristischem HTML-Parsing.
- Bevölkerungsdichte ist nur eine Kontextvariable.
- Bei acht Städten ist eine Rangkorrelation nur explorativ.
- Der Projektfokus liegt auf Big Data Engineering, nicht auf Kausalanalyse oder Machine Learning.

## Nächster Schritt
Die erzeugten Figuren und die deutschsprachige Storyline dienen als Grundlage für eine Präsentation von maximal zehn Minuten.